# Generation of new return period values per hydrobasin based on % change in peak flow

### Step 0: Import packages to work with and set up folder pathways

In [ ]:
from pathlib import Path
import numpy as np
import pandas
import scipy
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import matplotlib as mpl
import Robyn_river_floods
import re
from scipy.integrate import simpson
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.colors as mcolors
from matplotlib.lines import Line2D
from pathlib import Path

In [ ]:
# Implement this code in order to make any figures always use Times New Roman. Otherwise when you do it in the figures the matplotlib default can override it
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['font.size'] = 12

# Set figure titles to be the same
mpl.rcParams['axes.titlesize'] = 20
mpl.rcParams['axes.titleweight'] = 'bold'
mpl.rcParams['axes.titlepad'] = 20
mpl.rcParams['font.family'] = 'Times New Roman'

#### Output path

In [ ]:
output_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis/Processed_data")
output_folder = Path("/Users/robynhaggis/Documents/Geospatial_analysis/Processed_data/merged_with_hydrobasins")

#### Input paths

In [ ]:
# Import forests
forest_catchments_path = output_path / "catchment_forest_summary_with_percentages.csv"
forest_catchments = pandas.read_csv(forest_catchments_path)

In [ ]:
# Load interpolated peak flow reduction data
peak_flow_catchment_coverage_path = output_path / "interpolated_peak_flow_catchment_coverage.csv"
peak_flow_catchment_coverage = pandas.read_csv(peak_flow_catchment_coverage_path)

In [ ]:
jamaica_boundary_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis/Inputs/Boundaries/jamaica.gpkg")
jamaica_boundary = gpd.read_file(jamaica_boundary_path)
print(f"Original Jamaica boundary CRS: {jamaica_boundary.crs}")

In [ ]:
aggregated_summaries = Path("/Users/robynhaggis/Documents/Geospatial_analysis/Results/aggregated_summaries")
joined_networks_catchments_damages_for_interpolation_folder = Path("/Users/robynhaggis/Documents/Geospatial_analysis/Results/joined_networks_catchments_damages_for_interpolation")

In [ ]:
hydrobasins = Path("/Users/robynhaggis/Documents/Geospatial_analysis/Processed_data/HydroBASINS_Level12_Clipped_Jamaica.shp")
hydrobasins = gpd.read_file(hydrobasins)
print(hydrobasins.crs)

In [ ]:
future_damages_dir = aggregated_summaries.parent / "future_damages"
future_damages_dir.mkdir(parents=True, exist_ok=True)

### Interpolate between return periods

In [ ]:
# example data frame of pre-calculated total damages in each hydrobasin for a set of return periods
damages = pandas.DataFrame(
    data={
        "HYBAS_ID": [123, 124, 125],
        "rp0.001": [0,0,0],
        "rp2.0": [0,0,0],
        "rp20.0": [10,20,30]
    }
).set_index("HYBAS_ID")

# Define output return periods
rps_to_calculate = [2.0, 5.0, 10.0]

# Call the calculate_rp_maps function
interpolated_damages = Robyn_river_floods.interpolate_rp_damages(rps_to_calculate, damages)
interpolated_damages

### Step 4: Define how return period changes according to percentage change in peak flow

#### Interpolate to a full table of catchment forest percentages and peak flow reductions for RP5 and 100

In [ ]:
# Create the initial dataframe with the original column name

In [ ]:
Robyn_river_floods.peak_flow_reduction(forest_percentage_change=[0, 10, 100])

#### % Change in peak flow impact on return period

In [ ]:
Robyn_river_floods.rp_change_given_flow_reduction(reduction_percent=np.array([5.0, 100.0]), interp_rp=100.0)

### Forest catchment coverage

In [ ]:
# forest_catchments.current_forested_percentage, future_forest_including_agri_percentage
catchments = forest_catchments.set_index("HYBAS_ID")[['forest_flood_equivalent_percentage', 'afforestable_including_agriculture_percentage', 'total_future_forest_including_agri_percentage']]
catchments

In [ ]:
rp_cols = ["rp5.0","rp10.0","rp20.0","rp50.0","rp100.0"]
catchment_peak_flow_reduction = Robyn_river_floods.peak_flow_reduction(catchments.afforestable_including_agriculture_percentage) \
    [rp_cols] \
    .rename(columns={rp_col: f"pfr_{rp_col}" for rp_col in rp_cols})

catchments_with_rp_change = catchments.join(catchment_peak_flow_reduction)
catchments_with_rp_change

for rp_col in rp_cols:
    rp_change = Robyn_river_floods.rp_change_given_flow_reduction(reduction_percent=catchments_with_rp_change[f"pfr_{rp_col}"], interp_rp=float(rp_col.replace("rp", ""))) \
        [[rp_col]] \
        .rename(columns={rp_col: f"future_{rp_col}"})
    catchments_with_rp_change = catchments_with_rp_change.join(rp_change)

catchments_with_rp_change

## Future RP damages

In [ ]:
# Build aggregation dictionary and group by HYBAS_ID
cols_to_agg = [
    'fluvial__rp_20__rcp_baseline__epoch_2010__conf_None_amin',
    'fluvial__rp_20__rcp_baseline__epoch_2010__conf_None_mean',
    'fluvial__rp_20__rcp_baseline__epoch_2010__conf_None_amax',
    'fluvial__rp_50__rcp_baseline__epoch_2010__conf_None_amin',
    'fluvial__rp_50__rcp_baseline__epoch_2010__conf_None_mean',
    'fluvial__rp_50__rcp_baseline__epoch_2010__conf_None_amax',
    'fluvial__rp_100__rcp_baseline__epoch_2010__conf_None_amin', 
    'fluvial__rp_100__rcp_baseline__epoch_2010__conf_None_mean',
    'fluvial__rp_100__rcp_baseline__epoch_2010__conf_None_amax',
    'fluvial__rp_200__rcp_baseline__epoch_2010__conf_None_amin',
    'fluvial__rp_200__rcp_baseline__epoch_2010__conf_None_mean',
    'fluvial__rp_200__rcp_baseline__epoch_2010__conf_None_amax',
    'fluvial__rp_500__rcp_baseline__epoch_2010__conf_None_amin',
    'fluvial__rp_500__rcp_baseline__epoch_2010__conf_None_mean',
    'fluvial__rp_500__rcp_baseline__epoch_2010__conf_None_amax',
    'fluvial__rp_1500__rcp_baseline__epoch_2010__conf_None_amin',
    'fluvial__rp_1500__rcp_baseline__epoch_2010__conf_None_mean',
    'fluvial__rp_1500__rcp_baseline__epoch_2010__conf_None_amax'
]

# Loop over each .gpkg file in the folder
for gpkg_file in joined_networks_catchments_damages_for_interpolation_folder.glob("*.gpkg"):
    print(f"Processing file: {gpkg_file.name}")
    
    # Read the file into a GeoDataFrame
    gdf = gpd.read_file(gpkg_file)
    
    # Check if both HYBAS_ID and all the required columns are in the file
    if "HYBAS_ID" in gdf.columns and all(col in gdf.columns for col in cols_to_agg):
        # Extract only the HYBAS_ID and columns to aggregate
        df = gdf[["HYBAS_ID"] + cols_to_agg]
        
        # Group by HYBAS_ID and sum the specified columns
        aggregated_summary = df.groupby("HYBAS_ID", as_index=False)[cols_to_agg].sum()
        
        # Merge with the hydrobasins GeoDataFrame using the HYBAS_ID column
        merged_df = hydrobasins[['HYBAS_ID','geometry']].merge(aggregated_summary, on="HYBAS_ID", how="left").fillna(0)
        
        # Save the aggregated summary to a CSV file in the new folder
        output_file = aggregated_summaries / f"{gpkg_file.stem}_aggregated.csv"
        merged_df.drop(columns=['geometry']).to_csv(output_file, index=False)
        print(f"Saved aggregated summary to {output_file}")
        
        # Convert merged_df to a GeoDataFrame, preserving the geometry from hydrobasins
        merged_gdf = gpd.GeoDataFrame(merged_df, geometry='geometry', crs=hydrobasins.crs)
        
        # Save the merged GeoDataFrame as a new gpkg file
        output_file = output_folder / f"{gpkg_file.stem}_merged.gpkg"
        merged_gdf.to_file(output_file, driver="GPKG")
        print(f"Saved merged file to {output_file}")
    else:
        print(f"Skipping {gpkg_file.name}: required columns missing.")

In [ ]:
def calculate_future_sector_damages(sector_damages, catchments_with_rp_change):
    variant_dfs = []
    for variant in ['mean', 'amax', 'amin']:
        # select and rename your baseline damages to rpX.0, rpY.0, etc:
        selected_damages = Robyn_river_floods.select_damages(
            sector_damages, variant=variant
        ).reset_index()

        dfs = []
        for hybas_id, hybas in catchments_with_rp_change.iterrows():
            # pick out just this catchment’s baseline-damage row,
            # set HYBAS_ID into the index so it doesn’t get treated as an rp column
            hybas_damages = (
                selected_damages
                .loc[selected_damages.HYBAS_ID == hybas_id]
                .set_index("HYBAS_ID")
            )

            # baseline RPs to estimate
            interpolated_baseline_damages = Robyn_river_floods.interpolate_rp_damages(  # robyn import
                [5.0, 10.0],
                hybas_damages
            )

            baseline_damages = hybas_damages.join(interpolated_baseline_damages)

            future_rps = [float(colname.replace("future_rp", "")) for colname in catchments_with_rp_change.columns if "future_rp" in colname]
            
            # get the *future* RP values corresponding to each
            current_to_future_rps = {
                rp: hybas[f"future_rp{float(rp)}"]
                for rp in future_rps
            }
            # display(current_to_future_rps)

            # display(baseline_damages)

            future_rp_columns = [f"rp{rp}" for rp in future_rps]
            future_damages = baseline_damages[future_rp_columns].rename(columns={
                f"rp{current}": f"rp{future}"
                for current, future in current_to_future_rps.items()
            })

            extreme_damages = baseline_damages[["rp0.0001", "rp1000000000.0"]].copy()
            future_damages = future_damages.join(extreme_damages)
            future_damages_columns = sorted(
                future_damages.columns, 
                key=lambda c: float(c.replace("rp", ""))
            )
            future_damages = future_damages[future_damages_columns].copy()
            # at this point future damages contains columns for each of the adjusted future return periods
            # corresponding to the initial baseline set (e.g. rp6.4, rp12.0, rp24.1 corresponding to what was
            # rp5.0, rp10.0, rp20.0) - these could be used to calculate EAD directly, without the next bit
            # of interpolation.

            # could call Robyn_river_floods.calculate_ead here using future_damages
            
            # display(future_damages)
            # calculate EAD for this catchment - append to interpolated damages further down
            # 5) calculate EAD directly on these shifted‐RP columns
            # ead_values = Robyn_river_floods.calculate_ead(future_damages)
            # ead_colname = f"future__fluvial__ead__{variant}" 
            baseline_ead_values = Robyn_river_floods.calculate_ead(hybas_damages)
            baseline_ead_colname = f"baseline__fluvial__ead__{variant}" 


            # perform the interpolation 
            interpolated_hybas_damages = Robyn_river_floods.interpolate_rp_damages(
                future_rps,
                future_damages
            )
            # display(interpolated_hybas_damages)
            ead_values = Robyn_river_floods.calculate_ead(interpolated_hybas_damages)
            ead_colname = f"future__fluvial__ead__{variant}" 
            
            # now rename its *output* columns (one per rp_to_calculate)
            to_rename = {
                f"rp{rp}": f"future__fluvial__rp_{int(rp)}__future__fluvial__rp_{variant}"
                for rp in future_rps
            }
            interpolated_hybas_damages.rename(columns=to_rename, inplace=True)

            # pick out the interpolated future rp damages
            interpolated_hybas_damages = interpolated_hybas_damages[to_rename.values()]
            # add future ead as calculated above
            interpolated_hybas_damages[ead_colname] = ead_values

            # rename from rpXX columns to baseline...
            to_rename = {
                f"rp{rp}": f"baseline__fluvial__rp_{int(rp)}__baseline__fluvial__rp_{variant}"
                for rp in future_rps
            }
            baseline_damages_renamed = baseline_damages.rename(columns=to_rename)[to_rename.values()]
            baseline_damages_renamed[baseline_ead_colname] = baseline_ead_values
                
            # join the interpolated baseline rp damages
            interpolated_hybas_damages = pandas.concat([
                interpolated_hybas_damages,
                baseline_damages_renamed
            ], axis=1)

            # re‑attach the index as a column
            interpolated_hybas_damages["HYBAS_ID"] = hybas_id
            dfs.append(interpolated_hybas_damages)

        variant_damages = pandas.concat(dfs).set_index("HYBAS_ID")
        variant_dfs.append(variant_damages)

    # concatenate all three variants (mean, amax, amin) side‑by‑side
    return pandas.concat(variant_dfs, axis=1)
# future_damages#.loc[7120065500, "future__fluvial__rp_100__rcp_baseline__epoch_2010__conf_None_mean"]

In [ ]:
sector_damage_dfs = []
for sector_damages_csv in aggregated_summaries.glob("joined_*_aggregated.csv"): 
    sector = sector_damages_csv.name.replace("joined_", "").replace("_catchments_intersection_aggregated.csv","")
    print(sector)
    sector_damages = pandas.read_csv(sector_damages_csv)
    future_damages = calculate_future_sector_damages(sector_damages, catchments_with_rp_change)

    # both = sector_damages.set_index('HYBAS_ID').join(future_damages)
    future_damages.to_csv(future_damages_dir / f"future_damages_{sector}.csv")

In [ ]:
damage_dfs = []
for filename in future_damages_dir.glob("future_damages*.csv"):
    sector = filename.name.replace("future_damages_", "").replace(".csv","")
    print(sector)
    damages = pandas.read_csv(filename)
    damages['sector'] = sector
    damage_dfs.append(damages)
all_sector_damage = pandas.concat(damage_dfs)

In [ ]:
all_sector_damage.columns
list(all_sector_damage.columns)

In [ ]:
# Example usage: process a list of return periods
rps_to_process = [20, 50, 100]  # you can add more periods if needed
for variant in ['mean', 'amax', 'amin']:
    for rp in rps_to_process:
        baseline_col = f'baseline__fluvial__rp_{rp}__baseline__fluvial__rp_{variant}'
        future_col = f'future__fluvial__rp_{rp}__future__fluvial__rp_{variant}'
        avoided_col = f'avoided__fluvial__rp_{rp}__future__fluvial__rp_{variant}'
        all_sector_damage[avoided_col] = all_sector_damage[baseline_col] - all_sector_damage[future_col]
    all_sector_damage[f'avoided__fluvial__ead__{variant}'] = (
        all_sector_damage[f'baseline__fluvial__ead__{variant}']
        - all_sector_damage[f'future__fluvial__ead__{variant}']
    )
        
all_sector_damage.groupby('HYBAS_ID').sum()

In [ ]:
all_sector_damage.to_csv('all_sector_damage.csv', index=True)

In [ ]:
# --- 2) Compute avoided EAD per catchment ---
# assume all_sector_damage is already in your namespace
avoided_df = (
    all_sector_damage
    .groupby("HYBAS_ID")["avoided__fluvial__ead__mean"]
    .sum()
    .reset_index()
)

In [ ]:
# 2) make a copy
hydro = hydrobasins.copy()

# 3) now merge into 'hydro' without touching `hydrobasins`
avoided_df = all_sector_damage.groupby("HYBAS_ID")["avoided__fluvial__ead__mean"].sum().reset_index()
hydro["HYBAS_ID"] = hydro["HYBAS_ID"].astype(int)
avoided_df["HYBAS_ID"] = avoided_df["HYBAS_ID"].astype(int)

hydro = hydro.merge(avoided_df, on="HYBAS_ID", how="left")

In [ ]:
# fill any missing with zero (or leave as NaN if you prefer)
hydro["avoided__fluvial__ead__mean"] = hydro["avoided__fluvial__ead__mean"].fillna(0)

# --- 3) Build a color ramp from 0 → max(avoided) using OrRd ---
cmap = plt.colormaps["OrRd"]
vmax = hydro["avoided__fluvial__ead__mean"].max()
norm = mpl.colors.Normalize(vmin=0, vmax=vmax)

# store hex colors for each polygon
hydro["color"] = hydro["avoided__fluvial__ead__mean"]\
    .map(lambda val: mcolors.to_hex(cmap(norm(val))))

# --- 4) Helpers for scale bar & north arrow (copyed from your example) ---
def add_scale_bar(ax, length_km=20, location=(0.9, 0.79)):
    half = 0.05; x,y = location
    ax.plot([x-half, x+half], [y, y], transform=ax.transAxes, color='black', lw=2)
    for pos in (x-half, x, x+half):
        ax.plot([pos, pos], [y-0.005, y+0.005], transform=ax.transAxes, color='black', lw=2)
    ax.text(x-half, y-0.03, "0", transform=ax.transAxes, ha='center', va='center')
    ax.text(x,       y-0.03, f"{length_km//2}", transform=ax.transAxes, ha='center', va='center')
    ax.text(x+half,  y-0.03, f"{length_km}",    transform=ax.transAxes, ha='center', va='center')
    ax.text(x+half+0.02, y, "km", transform=ax.transAxes, ha='left', va='center')

def add_north_arrow(ax, location=(0.9, 0.85), size=0.05):
    x,y = location
    ax.annotate(
        "",
        xy=(x, y+size), xycoords='axes fraction',
        xytext=(x, y),   textcoords='axes fraction',
        arrowprops=dict(facecolor='black', edgecolor='black',
                        headwidth=10, headlength=15, width=5)
    )
    ax.text(x, y+size+0.02, "N", transform=ax.transAxes,
            ha='center', va='center', fontsize=12, fontweight='bold')

# --- 5) Plotting function ---
def plot_avoided_ead_map(gdf, title, out_dir, fname):
    fig, ax = plt.subplots(figsize=(18, 14), dpi=300)

    # fill catchments
    gdf.plot(ax=ax, color=gdf['color'], linewidth=0, alpha=0.8)
    # boundaries
    gdf.boundary.plot(ax=ax, edgecolor='black', linewidth=0.5)

    # continuous colorbar
    sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm._A = []
    cbar = fig.colorbar(sm, ax=ax, fraction=0.03, pad=0.04)
    cbar.set_label("Avoided Fluvial EAD (Mean)", family='Times New Roman')

    # optional: label each catchment by its HYBAS_ID
    for _, row in gdf.iterrows():
        pt = row.geometry.representative_point()
        ax.text(pt.x, pt.y, str(int(row.HYBAS_ID)),
                ha='center', va='center', fontsize=6, fontweight='bold')

    # legend stub for boundaries
    handle = Line2D([0], [0], color='black', lw=0.5, label='Catchment boundary')
    ax.legend(handles=[handle], title='Legend',
              bbox_to_anchor=(0.5, -0.1), loc='upper center',
              frameon=False, fontsize=12, title_fontsize=14,
              prop={'family':'Times New Roman'}).get_title().set_position((0,10))

    # cartographic elements
    add_scale_bar(ax)
    add_north_arrow(ax)

    ax.set_axis_off()
    plt.title(title, fontsize=20, fontweight='bold',
              fontname='Times New Roman', pad=20)
    plt.tight_layout()

    # save outputs
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_dir / f"{fname}.png", dpi=300, bbox_inches='tight')
    fig.savefig(out_dir / f"{fname}.pdf", bbox_inches='tight')
    print(f"Saved {out_dir/fname}.png and {out_dir/fname}.pdf")

    plt.show()

# --- 6) Run it ---
plot_avoided_ead_map(
    hydro,
    title="Avoided Fluvial EAD (Mean) by Catchment in Jamaica",
    out_dir="path/to/output_dir",
    fname="Figure_avoided_fluvial_ead_mean"
)

In [ ]:
if 'sector' in all_sector_damage.columns:
    print(all_sector_damage['sector'].unique())
else:
    print("No 'sector' column present")

In [ ]:
# --- 0) your sector → category mapping ---
sector_map = {
    'buildings_areas':             'Buildings',
    'roads_nodes':                 'Roads',
    'roads_edges':                 'Roads',
    'rail_nodes':                  'Rail',
    'rail_edges':                  'Rail',
    'ports_areas':                 'Ports',
    'airports_areas':              'Airports',
    'electricity_nodes':           'Energy',
    'irrigation_nodes':            'Water',
    'irrigation_edges':            'Water',
    'pipelines_edges':             'Water',
    'wastewater_nodes':            'Water',
    'potable_facilities_nodes':    'Water'
}

# --- 1) Assign the new category and compute avoided mean EAD ---
df = all_sector_damage.copy()

# create a clean category column
df['category'] = df['sector'].map(sector_map)

# compute avoided mean EAD if you haven't already
df['avoided_ead_mean'] = (
    df['baseline__fluvial__ead__mean']
  - df['future__fluvial__ead__mean']
)

# aggregate by category + catchment
avoided = (
    df
    .groupby(['category', 'HYBAS_ID'])['avoided_ead_mean']
    .sum()
    .reset_index()
)
def plot_category_map(cat_gdf, category_name, out_dir):
    cmap = plt.colormaps["OrRd"]
    vmax = cat_gdf["avoided_ead_mean"].max()
    norm = mpl.colors.Normalize(vmin=0, vmax=vmax)

    cat_gdf["color"] = cat_gdf["avoided_ead_mean"].map(
        lambda v: mcolors.to_hex(cmap(norm(v)))
    )

    fig, ax = plt.subplots(figsize=(12, 10), dpi=200)
    cat_gdf.plot(ax=ax, color=cat_gdf["color"], linewidth=0, alpha=0.8)
    cat_gdf.boundary.plot(ax=ax, edgecolor="black", linewidth=0.4)

    sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm._A = []
    cbar = fig.colorbar(sm, ax=ax, fraction=0.03, pad=0.02)
    cbar.set_label(f"Avoided Fluvial EAD (Mean) — {category_name}", fontsize=10)

    ax.set_title(f"{category_name}: Avoided Fluvial EAD (Mean)", fontsize=14)
    ax.set_axis_off()
    plt.tight_layout()

    # Save to disk
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_dir / f"avoided_ead_{category_name.lower()}.png", dpi=300)

    # **Show it inline in the notebook:**
    plt.show()
for category in avoided['category'].unique():
    # ... merge data into `hydro` as before ...
    plot_category_map(hydro, category, output_folder)

In [ ]:

# 1) Your mapping and aggregation (if you haven’t run it already)
sector_map = {
    'buildings_areas':          'Buildings',
    'roads_nodes':              'Roads',
    'roads_edges':              'Roads',
    'rail_nodes':               'Rail',
    'rail_edges':               'Rail',
    'ports_areas':              'Ports',
    'airports_areas':           'Airports',
    'electricity_nodes':        'Energy',
    'irrigation_nodes':         'Water',
    'irrigation_edges':         'Water',
    'pipelines_edges':          'Water',
    'wastewater_nodes':         'Water',
    'potable_facilities_nodes': 'Water'
}

df = all_sector_damage.copy()
df['category'] = df['sector'].map(sector_map)
df['avoided_ead_mean'] = (
    df['baseline__fluvial__ead__mean']
  - df['future__fluvial__ead__mean']
)
avoided = (
    df
    .groupby(['category','HYBAS_ID'])['avoided_ead_mean']
    .sum()
    .reset_index()
)


# 3) Plotting function
def plot_category_map(gdf, category_name):
    cmap = plt.colormaps["OrRd"]
    vmax = gdf["avoided_ead_mean"].max()
    norm = mpl.colors.Normalize(vmin=0, vmax=vmax)

    fig, ax = plt.subplots(figsize=(8,6))
    gdf.plot(
        ax=ax,
        column="avoided_ead_mean",
        cmap=cmap,
        norm=norm,
        linewidth=0.3,
        edgecolor='black',
        legend=True,
        legend_kwds={
            'label': f"Avoided EAD (Mean) — {category_name}",
            'shrink': 0.6
        }
    )
    ax.set_title(f"{category_name}: Avoided Fluvial EAD (Mean)", fontsize=14)
    ax.set_axis_off()
    plt.tight_layout()
    plt.show()

# 4) Loop over categories
sector_order = ['Buildings','Roads','Rail','Ports','Airports','Energy','Water']
for category in sector_order:
    # filter the aggregated data
    df_cat = avoided[avoided['category']==category].copy()
    df_cat["HYBAS_ID"] = df_cat["HYBAS_ID"].astype(int)

    # merge into a fresh copy of the geometry
    hydro = hydrobasins.copy()
    hydro = hydro.merge(df_cat, on="HYBAS_ID", how="left")
    hydro["avoided_ead_mean"] = hydro["avoided_ead_mean"].fillna(0)

    # draw it
    plot_category_map(hydro, category)

In [ ]:
# fill any missing with zero (or leave as NaN if you prefer)
hydro["avoided__fluvial__ead__mean"] = hydro["avoided__fluvial__ead__mean"].fillna(0)

# --- 3) Build a color ramp from 0 → max(avoided) using OrRd ---
cmap = plt.colormaps["OrRd"]
vmax = hydro["avoided__fluvial__ead__mean"].max()
norm = mpl.colors.Normalize(vmin=0, vmax=vmax)

# store hex colors for each polygon
hydro["color"] = hydro["avoided__fluvial__ead__mean"]\
    .map(lambda val: mcolors.to_hex(cmap(norm(val))))

# --- 4) Helpers for scale bar & north arrow (copyed from your example) ---
def add_scale_bar(ax, length_km=20, location=(0.9, 0.79)):
    half = 0.05; x,y = location
    ax.plot([x-half, x+half], [y, y], transform=ax.transAxes, color='black', lw=2)
    for pos in (x-half, x, x+half):
        ax.plot([pos, pos], [y-0.005, y+0.005], transform=ax.transAxes, color='black', lw=2)
    ax.text(x-half, y-0.03, "0", transform=ax.transAxes, ha='center', va='center')
    ax.text(x,       y-0.03, f"{length_km//2}", transform=ax.transAxes, ha='center', va='center')
    ax.text(x+half,  y-0.03, f"{length_km}",    transform=ax.transAxes, ha='center', va='center')
    ax.text(x+half+0.02, y, "km", transform=ax.transAxes, ha='left', va='center')

def add_north_arrow(ax, location=(0.9, 0.85), size=0.05):
    x,y = location
    ax.annotate(
        "",
        xy=(x, y+size), xycoords='axes fraction',
        xytext=(x, y),   textcoords='axes fraction',
        arrowprops=dict(facecolor='black', edgecolor='black',
                        headwidth=10, headlength=15, width=5)
    )
    ax.text(x, y+size+0.02, "N", transform=ax.transAxes,
            ha='center', va='center', fontsize=12, fontweight='bold')

# --- 5) Plotting function ---
def plot_avoided_ead_map(gdf, title, out_dir, fname):
    fig, ax = plt.subplots(figsize=(18, 14), dpi=300)

    # fill catchments
    gdf.plot(ax=ax, color=gdf['color'], linewidth=0, alpha=0.8)
    # boundaries
    gdf.boundary.plot(ax=ax, edgecolor='black', linewidth=0.5)

    # continuous colorbar
    sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm._A = []
    cbar = fig.colorbar(sm, ax=ax, fraction=0.03, pad=0.04)
    cbar.set_label("Avoided Fluvial EAD (Mean)", family='Times New Roman')

    # optional: label each catchment by its HYBAS_ID
    for _, row in gdf.iterrows():
        pt = row.geometry.representative_point()
        ax.text(pt.x, pt.y, str(int(row.HYBAS_ID)),
                ha='center', va='center', fontsize=6, fontweight='bold')

    # legend stub for boundaries
    handle = Line2D([0], [0], color='black', lw=0.5, label='Catchment boundary')
    ax.legend(handles=[handle], title='Legend',
              bbox_to_anchor=(0.5, -0.1), loc='upper center',
              frameon=False, fontsize=12, title_fontsize=14,
              prop={'family':'Times New Roman'}).get_title().set_position((0,10))

    # cartographic elements
    add_scale_bar(ax)
    add_north_arrow(ax)

    ax.set_axis_off()
    plt.title(title, fontsize=20, fontweight='bold',
              fontname='Times New Roman', pad=20)
    plt.tight_layout()

    # save outputs
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_dir / f"{fname}.png", dpi=300, bbox_inches='tight')
    fig.savefig(out_dir / f"{fname}.pdf", bbox_inches='tight')
    print(f"Saved {out_dir/fname}.png and {out_dir/fname}.pdf")

    plt.show()

# --- 6) Run it ---
plot_avoided_ead_map(
    hydro,
    title="Avoided Fluvial EAD (Mean) by Catchment in Jamaica",
    out_dir="path/to/output_dir",
    fname="Figure_avoided_fluvial_ead_mean"
)

In [ ]:
# Ccombine them into a single DataFrame
combined_sector_damage = pandas.concat(dfs, ignore_index=True)
combined_sector_damage.head()

In [ ]:
combined_sector_damage.to_csv(future_damages_dir / "combined_sector_damage.csv")
all_sector_damage.to_csv(future_damages_dir / "all_sector_damage.csv")

In [ ]:
display(all_sector_damage[[ "HYBAS_ID", fluvial_col, future_col ]].sample(10))

In [ ]:
print(all_sector_damage[fluvial_col].describe())
print(all_sector_damage[future_col].describe())

In [ ]:
print(all_sector_damage.loc[all_sector_damage[fluvial_col] != 0, [ "HYBAS_ID", fluvial_col, future_col ]].sample(10))

In [ ]:
# Pivot the combined DataFrame so that each HYBAS_ID has separate columns for each rp's damages.
damage_pivot = combined_sector_damage.pivot(index="HYBAS_ID", columns="rp", values="damages")

In [ ]:
print("Return period columns:", damage_pivot.columns)

In [ ]:
# Rename columns to have the format "rp20", "rp50", "rp100", etc.
damage_pivot.columns = [f"rp{int(c)}" for c in damage_pivot.columns]

In [ ]:
# Calculate EAD using the Simpson integration function.
damage_pivot["ead"] = calculate_ead(damage_pivot)

# View the result for a couple of catchments
damage_pivot.head()

In [ ]:
# Pivot for baseline damages (damages)
pivot_damages = combined_sector_damage.pivot(index="HYBAS_ID", columns="rp", values="damages")
pivot_damages.columns = [f"rp{int(c)}" for c in pivot_damages.columns]
print("Return period columns in baseline damage pivot:", pivot_damages.columns.tolist())

# Pivot for damages with NBS
pivot_damages_with_nbs = combined_sector_damage.pivot(index="HYBAS_ID", columns="rp", values="damages_with_nbs")
pivot_damages_with_nbs.columns = [f"rp{int(c)}" for c in pivot_damages_with_nbs.columns]
print("Return period columns in damages_with_nbs pivot:", pivot_damages_with_nbs.columns.tolist())

# Calculate the EAD values (integration over the entire range of available RP columns)
pivot_damages["ead_baseline"] = calculate_ead(pivot_damages)
pivot_damages_with_nbs["ead_with_nbs"] = calculate_ead(pivot_damages_with_nbs)

# Merge the two EAD results using the index (HYBAS_ID)
ead_comparison = pivot_damages[["ead_baseline"]].join(pivot_damages_with_nbs[["ead_with_nbs"]])
ead_comparison["ead_difference"] = ead_comparison["ead_baseline"] - ead_comparison["ead_with_nbs"]

# Display the results with a note about the RP range used
print("The EAD values were calculated over these return periods:")
print(pivot_damages.columns.tolist())  # This shows the RP columns used in the integration
display(ead_comparison.head())

print("Return period columns in baseline damage pivot:", pivot_damages.columns.tolist())

# Extract just the rp columns (i.e. those that start with "rp") from the pivot
rp_columns_baseline = [col for col in pivot_damages.columns if col.startswith("rp") and col != "ead_baseline"]
print("Return periods used in baseline integration:", rp_columns_baseline)



In [ ]:
# Pivot for baseline damages (damages)
pivot_damages = combined_sector_damage.pivot(index="HYBAS_ID", columns="rp", values="damages")
# Rename columns to a standard format like "rp20", "rp50", etc.
pivot_damages.columns = [f"rp{int(c)}" for c in pivot_damages.columns]
print("Baseline damages pivot columns:", pivot_damages.columns.tolist())

# Pivot for damages with NBS
pivot_damages_with_nbs = combined_sector_damage.pivot(index="HYBAS_ID", columns="rp", values="damages_with_nbs")
# Rename columns similarly.
pivot_damages_with_nbs.columns = [f"rp{int(c)}" for c in pivot_damages_with_nbs.columns]
print("Damages with NBS pivot columns:", pivot_damages_with_nbs.columns.tolist())

In [ ]:
# Calculate EAD (Expected Annual Damage) for each damage type:
pivot_damages["ead_baseline"] = Robyn_river_floods.calculate_ead(pivot_damages)
pivot_damages_with_nbs["ead_with_nbs"] = Robyn_river_floods.calculate_ead(pivot_damages_with_nbs)

# Examine the results:
display(pivot_damages.head())
display(pivot_damages_with_nbs.head())

In [ ]:
rp20_all_sector = all_sector_damage[['HYBAS_ID', 'fluvial__rp_20__rcp_baseline__epoch_2010__conf_None_mean',
                   'future__fluvial__rp_20__rcp_baseline__epoch_2010__conf_None_mean']].groupby('HYBAS_ID').sum()
rp20_all_sector = hydrobasins[['HYBAS_ID','geometry']].set_index('HYBAS_ID').join(rp20_all_sector)
rp20_all_sector['avoided__fluvial__rp_20'] = (
    rp20_all_sector['fluvial__rp_20__rcp_baseline__epoch_2010__conf_None_mean'] - 
    rp20_all_sector['future__fluvial__rp_20__rcp_baseline__epoch_2010__conf_None_mean']
)
rp20_all_sector['rp'] = 20
rp20_all_sector = rp20_all_sector.reset_index()
rp20_all_sector.columns = ["HYBAS_ID","geometry","damages","damages_with_nbs","avoided_damages","rp"]
rp20_all_sector


In [ ]:
# Drop geometry for grouping and summing
rp20_all_sector_numeric = rp20_all_sector.drop(columns="geometry")

# Group by HYBAS_ID and rp using sum
rp20_grouped = rp20_all_sector_numeric.groupby(["HYBAS_ID", "rp"]).sum().reset_index()

# Pivot the grouped DataFrame
rp20_pivot = rp20_grouped.pivot(index="HYBAS_ID", columns="rp").replace(float("NaN"), 0)
rp20_pivot.columns = [f"rp{int(rp)}" for _, rp in rp20_pivot.columns]
rp20_pivot.head(2)

# Optionally, join the geometry back if needed:
geometry = rp20_all_sector.set_index("HYBAS_ID")["geometry"]
rp20_final = geometry.to_frame().join(rp20_pivot)

In [ ]:
# def calculate_ead(df):
#     # Filter only columns that match "rp" followed by digits.
#     rp_cols = [col for col in df.columns if re.match(r'^rp\d+$', col)]
#     # Sort the columns using the reciprocal of the integer value after "rp"
#     rp_cols = sorted(rp_cols, key=lambda col: 1 / int(col.replace("rp", "")))
#     rps = np.array([int(col.replace("rp", "")) for col in rp_cols])
#     probabilities = 1 / rps
#     rp_damages = df[rp_cols]
#     return simpson(rp_damages, x=probabilities, axis=1)
    
rp20_all_sector["ead"] = Robyn_river_floods.calculate_ead(rp20_all_sector)
rp20_all_sector.head(2)

In [ ]:
rp20_all_sector.plot(column='avoided__fluvial__rp_20', legend=True, cmap='magma_r')

In [ ]:
rp50_all_sector = all_sector_damage[['HYBAS_ID', 'fluvial__rp_50__rcp_baseline__epoch_2010__conf_None_mean',
                   'future__fluvial__rp_50__rcp_baseline__epoch_2010__conf_None_mean']].groupby('HYBAS_ID').sum()
rp50_all_sector = hydrobasins[['HYBAS_ID','geometry']].set_index('HYBAS_ID').join(rp50_all_sector)
rp50_all_sector['avoided__fluvial__rp_50'] = (
    rp50_all_sector['fluvial__rp_50__rcp_baseline__epoch_2010__conf_None_mean'] - 
    rp50_all_sector['future__fluvial__rp_50__rcp_baseline__epoch_2010__conf_None_mean']
)
rp50_all_sector['rp'] = 50
rp50_all_sector = rp50_all_sector.reset_index()
rp50_all_sector.columns = ["HYBAS_ID","geometry","damages","damages_with_nbs","avoided_damages","rp"]
rp50_all_sector

In [ ]:
rp50_all_sector.plot(column='avoided__fluvial__rp_50', legend=True, cmap='magma_r')

In [ ]:
rp100_all_sector = all_sector_damage[['HYBAS_ID', 'fluvial__rp_100__rcp_baseline__epoch_2010__conf_None_mean',
                   'future__fluvial__rp_100__rcp_baseline__epoch_2010__conf_None_mean']].groupby('HYBAS_ID').sum()
rp100_all_sector = hydrobasins[['HYBAS_ID','geometry']].set_index('HYBAS_ID').join(rp100_all_sector)
rp100_all_sector['avoided__fluvial__rp_100'] = (
    rp100_all_sector['fluvial__rp_100__rcp_baseline__epoch_2010__conf_None_mean'] - 
    rp100_all_sector['future__fluvial__rp_100__rcp_baseline__epoch_2010__conf_None_mean']
)
rp100_all_sector['rp'] = 100
rp100_all_sector = rp100_all_sector.reset_index()
rp100_all_sector.columns = ["HYBAS_ID","geometry","damages","damages_with_nbs","avoided_damages","rp"]
rp100_all_sector

In [ ]:
damages_df = pandas.concat([rp20_all_sector,rp50_all_sector,rp100_all_sector],axis=0,ignore_index=True)
damages_df
historical = damages_df[["HYBAS_ID", "rp", "damages"]]

In [ ]:
rp100_all_sector.plot(column='avoided__fluvial__rp_100', legend=True, cmap='magma_r')

In [ ]:
# Create a table with the desired columns and reset the index to include HYBAS_ID as a column
table_df = rp100_all_sector[['fluvial__rp_100__rcp_baseline__epoch_2010__conf_None_mean',
                             'future__fluvial__rp_100__rcp_baseline__epoch_2010__conf_None_mean',
                             'avoided__fluvial__rp_100']].reset_index()

# Display the table (showing the first few rows)
display(table_df.head())

# If you're using Jupyter, you can also simply put 'table_df' in a cell to display it nicely:
table_df

# Create a table with the desired columns and reset the index to include HYBAS_ID as a column
table_df = rp100_all_sector[['fluvial__rp_100__rcp_baseline__epoch_2010__conf_None_mean',
                             'future__fluvial__rp_100__rcp_baseline__epoch_2010__conf_None_mean',
                             'avoided__fluvial__rp_100']].reset_index()

# Calculate the avoided percentage relative to the baseline
table_df['avoided_pct'] = (table_df['avoided__fluvial__rp_100'] / 
                           table_df['future__fluvial__rp_100__rcp_baseline__epoch_2010__conf_None_mean']) * 100

# Optionally, round the percentage values to 2 decimal places
table_df['avoided_pct'] = table_df['avoided_pct'].round(2)

# Display the updated table
print(table_df.head())
table_df

table_df['rp'] = 100

In [ ]:
# Define the transport sectors
transport_sectors = [
    "airports_areas",
    "roads_edges",
    "rail_nodes",
    "rail_edges",
    "roads_nodes",
    "ports_areas"
]

# Process only transport sector CSV files
for sector_damages_csv in aggregated_summaries.glob("joined_*_aggregated.csv"):
    # Extract sector name from filename
    sector = sector_damages_csv.name.replace("joined_", "").replace("_catchments_intersection_aggregated.csv", "")
    # Only process if sector is in the transport list
    if sector in transport_sectors:
        print(f"Processing transport sector: {sector}")
        sector_damages = pandas.read_csv(sector_damages_csv)
        future_damages = calculate_future_sector_damages(sector_damages)
        both = sector_damages.set_index('HYBAS_ID').join(future_damages)
        both.to_csv(future_damages_dir / f"future_damages_{sector}.csv")

# Now, read back the processed CSV files for the transport sectors
transport_damage_dfs = []
for filename in future_damages_dir.glob("future_damages*.csv"):
    # Extract sector name from filename
    sector = filename.name.replace("future_damages_", "").replace(".csv", "")
    # Only load the file if it belongs to a transport sector
    if sector in transport_sectors:
        print(f"Loading transport sector: {sector}")
        damages = pandas.read_csv(filename)
        damages['sector'] = sector
        transport_damage_dfs.append(damages)

# Concatenate all transport sector damage dataframes into one
all_transport_damage = pandas.concat(transport_damage_dfs)

# Aggregate the data by 'HYBAS_ID' and calculate the sum of damages
transport_rp100 = all_transport_damage[
    ['HYBAS_ID', 
     'fluvial__rp_100__rcp_baseline__epoch_2010__conf_None_mean',
     'future__fluvial__rp_100__rcp_baseline__epoch_2010__conf_None_mean']
].groupby('HYBAS_ID').sum()

# Join with geometry information from hydrobasins dataframe
transport_rp100 = hydrobasins[['HYBAS_ID', 'geometry']].set_index('HYBAS_ID').join(transport_rp100)

# Calculate the avoided damages for the transport sector
transport_rp100['avoided__fluvial__rp_100'] = (
    transport_rp100['fluvial__rp_100__rcp_baseline__epoch_2010__conf_None_mean'] - 
    transport_rp100['future__fluvial__rp_100__rcp_baseline__epoch_2010__conf_None_mean']
)

In [ ]:
transport_rp100.plot(column='avoided__fluvial__rp_100', legend=True, cmap='magma_r')

In [ ]:
# Define the transport sectors
transport_sectors = [
    "airports_areas",
    "roads_edges",
    "rail_nodes",
    "rail_edges",
    "roads_nodes",
    "ports_areas"
]

# Process only transport sector CSV files
for sector_damages_csv in aggregated_summaries.glob("joined_*_aggregated.csv"):
    # Extract sector name from filename
    sector = sector_damages_csv.name.replace("joined_", "").replace("_catchments_intersection_aggregated.csv", "")
    # Only process if sector is in the transport list
    if sector in transport_sectors:
        print(f"Processing transport sector: {sector}")
        sector_damages = pandas.read_csv(sector_damages_csv)
        future_damages = calculate_future_sector_damages(sector_damages)
        both = sector_damages.set_index('HYBAS_ID').join(future_damages)
        both.to_csv(future_damages_dir / f"future_damages_{sector}.csv")

# Now, read back the processed CSV files for the transport sectors
transport_damage_dfs = []
for filename in future_damages_dir.glob("future_damages*.csv"):
    # Extract sector name from filename
    sector = filename.name.replace("future_damages_", "").replace(".csv", "")
    # Only load the file if it belongs to a transport sector
    if sector in transport_sectors:
        print(f"Loading transport sector: {sector}")
        damages = pandas.read_csv(filename)
        damages['sector'] = sector
        transport_damage_dfs.append(damages)

# Concatenate all transport sector damage dataframes into one
all_transport_damage = pandas.concat(transport_damage_dfs)

# Aggregate the data by 'HYBAS_ID' and calculate the sum of damages
transport_rp50 = all_transport_damage[
    ['HYBAS_ID', 
     'fluvial__rp_50__rcp_baseline__epoch_2010__conf_None_mean',
     'future__fluvial__rp_50__rcp_baseline__epoch_2010__conf_None_mean']
].groupby('HYBAS_ID').sum()

# Join with geometry information from hydrobasins dataframe
transport_rp50 = hydrobasins[['HYBAS_ID', 'geometry']].set_index('HYBAS_ID').join(transport_rp50)

# Calculate the avoided damages for the transport sector
transport_rp50['avoided__fluvial__rp_50'] = (
    transport_rp50['fluvial__rp_50__rcp_baseline__epoch_2010__conf_None_mean'] - 
    transport_rp50['future__fluvial__rp_50__rcp_baseline__epoch_2010__conf_None_mean']
)

transport_rp50.plot(column='avoided__fluvial__rp_50', legend=True, cmap='magma_r')

In [ ]:
# Define the transport sectors
transport_sectors = [
    "airports_areas",
    "roads_edges",
    "rail_nodes",
    "rail_edges",
    "roads_nodes",
    "ports_areas"
]

# Process only transport sector CSV files
for sector_damages_csv in aggregated_summaries.glob("joined_*_aggregated.csv"):
    # Extract sector name from filename
    sector = sector_damages_csv.name.replace("joined_", "").replace("_catchments_intersection_aggregated.csv", "")
    # Only process if sector is in the transport list
    if sector in transport_sectors:
        print(f"Processing transport sector: {sector}")
        sector_damages = pandas.read_csv(sector_damages_csv)
        future_damages = calculate_future_sector_damages(sector_damages)
        both = sector_damages.set_index('HYBAS_ID').join(future_damages)
        both.to_csv(future_damages_dir / f"future_damages_{sector}.csv")

# Now, read back the processed CSV files for the transport sectors
transport_damage_dfs = []
for filename in future_damages_dir.glob("future_damages*.csv"):
    # Extract sector name from filename
    sector = filename.name.replace("future_damages_", "").replace(".csv", "")
    # Only load the file if it belongs to a transport sector
    if sector in transport_sectors:
        print(f"Loading transport sector: {sector}")
        damages = pandas.read_csv(filename)
        damages['sector'] = sector
        transport_damage_dfs.append(damages)

# Concatenate all transport sector damage dataframes into one
all_transport_damage = pandas.concat(transport_damage_dfs)

# Aggregate the data by 'HYBAS_ID' and calculate the sum of damages
transport_rp20 = all_transport_damage[
    ['HYBAS_ID', 
     'fluvial__rp_20__rcp_baseline__epoch_2010__conf_None_mean',
     'future__fluvial__rp_20__rcp_baseline__epoch_2010__conf_None_mean']
].groupby('HYBAS_ID').sum()

# Join with geometry information from hydrobasins dataframe
transport_rp20 = hydrobasins[['HYBAS_ID', 'geometry']].set_index('HYBAS_ID').join(transport_rp20)

# Calculate the avoided damages for the transport sector
transport_rp20['avoided__fluvial__rp_20'] = (
    transport_rp20['fluvial__rp_20__rcp_baseline__epoch_2010__conf_None_mean'] - 
    transport_rp20['future__fluvial__rp_20__rcp_baseline__epoch_2010__conf_None_mean']
)

transport_rp20.plot(column='avoided__fluvial__rp_20', legend=True, cmap='magma_r')

In [ ]:
# Define the energy sectors
energy_sectors = [
    "electricity_nodes",
]

# Process only energy sector CSV files
for sector_damages_csv in aggregated_summaries.glob("joined_*_aggregated.csv"):
    # Extract sector name from filename
    sector = sector_damages_csv.name.replace("joined_", "").replace("_catchments_intersection_aggregated.csv", "")
    # Only process if sector is in the transport list
    if sector in energy_sectors:
        print(f"Processing energy sector: {sector}")
        sector_damages = pandas.read_csv(sector_damages_csv)
        future_damages = calculate_future_sector_damages(sector_damages)
        both = sector_damages.set_index('HYBAS_ID').join(future_damages)
        both.to_csv(future_damages_dir / f"future_damages_{sector}.csv")

# Now, read back the processed CSV files for the transport sectors
energy_damage_dfs = []
for filename in future_damages_dir.glob("future_damages*.csv"):
    # Extract sector name from filename
    sector = filename.name.replace("future_damages_", "").replace(".csv", "")
    # Only load the file if it belongs to a transport sector
    if sector in energy_sectors:
        print(f"Loading energy sector: {sector}")
        damages = pandas.read_csv(filename)
        damages['sector'] = sector
        energy_damage_dfs.append(damages)

# Concatenate all transport sector damage dataframes into one
all_energy_damage = pandas.concat(energy_damage_dfs)

# Aggregate the data by 'HYBAS_ID' and calculate the sum of damages
energy_rp100 = all_energy_damage[
    ['HYBAS_ID', 
     'fluvial__rp_100__rcp_baseline__epoch_2010__conf_None_mean',
     'future__fluvial__rp_100__rcp_baseline__epoch_2010__conf_None_mean']
].groupby('HYBAS_ID').sum()

# Join with geometry information from hydrobasins dataframe
energy_rp100 = hydrobasins[['HYBAS_ID', 'geometry']].set_index('HYBAS_ID').join(energy_rp100)

# Calculate the avoided damages for the energy sector
energy_rp100['avoided__fluvial__rp_100'] = (
    energy_rp100['fluvial__rp_100__rcp_baseline__epoch_2010__conf_None_mean'] - 
    energy_rp100['future__fluvial__rp_100__rcp_baseline__epoch_2010__conf_None_mean']
)

energy_rp100.plot(column='avoided__fluvial__rp_100', legend=True, cmap='magma_r')

In [ ]:
# Define the energy sectors
energy_sectors = [
    "electricity_nodes",
]

# Process only energy sector CSV files
for sector_damages_csv in aggregated_summaries.glob("joined_*_aggregated.csv"):
    # Extract sector name from filename
    sector = sector_damages_csv.name.replace("joined_", "").replace("_catchments_intersection_aggregated.csv", "")
    # Only process if sector is in the transport list
    if sector in energy_sectors:
        print(f"Processing energy sector: {sector}")
        sector_damages = pandas.read_csv(sector_damages_csv)
        future_damages = calculate_future_sector_damages(sector_damages)
        both = sector_damages.set_index('HYBAS_ID').join(future_damages)
        both.to_csv(future_damages_dir / f"future_damages_{sector}.csv")

# Now, read back the processed CSV files for the energy sectors
energy_damage_dfs = []
for filename in future_damages_dir.glob("future_damages*.csv"):
    # Extract sector name from filename
    sector = filename.name.replace("future_damages_", "").replace(".csv", "")
    # Only load the file if it belongs to a transport sector
    if sector in energy_sectors:
        print(f"Loading energy sector: {sector}")
        damages = pandas.read_csv(filename)
        damages['sector'] = sector
        energy_damage_dfs.append(damages)

# Concatenate all energy sector damage dataframes into one
all_energy_damage = pandas.concat(energy_damage_dfs)

# Aggregate the data by 'HYBAS_ID' and calculate the sum of damages
energy_rp50 = all_energy_damage[
    ['HYBAS_ID', 
     'fluvial__rp_50__rcp_baseline__epoch_2010__conf_None_mean',
     'future__fluvial__rp_50__rcp_baseline__epoch_2010__conf_None_mean']
].groupby('HYBAS_ID').sum()

# Join with geometry information from hydrobasins dataframe
energy_rp50 = hydrobasins[['HYBAS_ID', 'geometry']].set_index('HYBAS_ID').join(energy_rp50)

# Calculate the avoided damages for the energy sector
energy_rp50['avoided__fluvial__rp_50'] = (
    energy_rp50['fluvial__rp_50__rcp_baseline__epoch_2010__conf_None_mean'] - 
    energy_rp50['future__fluvial__rp_50__rcp_baseline__epoch_2010__conf_None_mean']
)

energy_rp50.plot(column='avoided__fluvial__rp_50', legend=True, cmap='magma_r')

In [ ]:
# Define the energy sectors
energy_sectors = [
    "electricity_nodes",
]

# Process only energy sector CSV files
for sector_damages_csv in aggregated_summaries.glob("joined_*_aggregated.csv"):
    # Extract sector name from filename
    sector = sector_damages_csv.name.replace("joined_", "").replace("_catchments_intersection_aggregated.csv", "")
    # Only process if sector is in the transport list
    if sector in energy_sectors:
        print(f"Processing energy sector: {sector}")
        sector_damages = pandas.read_csv(sector_damages_csv)
        future_damages = calculate_future_sector_damages(sector_damages)
        both = sector_damages.set_index('HYBAS_ID').join(future_damages)
        both.to_csv(future_damages_dir / f"future_damages_{sector}.csv")

# Now, read back the processed CSV files for the energy sectors
energy_damage_dfs = []
for filename in future_damages_dir.glob("future_damages*.csv"):
    # Extract sector name from filename
    sector = filename.name.replace("future_damages_", "").replace(".csv", "")
    # Only load the file if it belongs to a transport sector
    if sector in energy_sectors:
        print(f"Loading energy sector: {sector}")
        damages = pandas.read_csv(filename)
        damages['sector'] = sector
        energy_damage_dfs.append(damages)

# Concatenate all energy sector damage dataframes into one
all_energy_damage = pandas.concat(energy_damage_dfs)

# Aggregate the data by 'HYBAS_ID' and calculate the sum of damages
energy_rp20 = all_energy_damage[
    ['HYBAS_ID', 
     'fluvial__rp_20__rcp_baseline__epoch_2010__conf_None_mean',
     'future__fluvial__rp_20__rcp_baseline__epoch_2010__conf_None_mean']
].groupby('HYBAS_ID').sum()

# Join with geometry information from hydrobasins dataframe
energy_rp20 = hydrobasins[['HYBAS_ID', 'geometry']].set_index('HYBAS_ID').join(energy_rp20)

# Calculate the avoided damages for the energy sector
energy_rp20['avoided__fluvial__rp_20'] = (
    energy_rp20['fluvial__rp_20__rcp_baseline__epoch_2010__conf_None_mean'] - 
    energy_rp20['future__fluvial__rp_20__rcp_baseline__epoch_2010__conf_None_mean']
)

energy_rp20.plot(column='avoided__fluvial__rp_20', legend=True, cmap='magma_r')

In [ ]:
# Define the water sectors
water_sectors = [
    "irrigation_edges",
    "pipelines_edges",
    "potable_facilities_nodes",
    "irrigation_nodes",
    "wastewater_nodes"
]

# Process only water sector CSV files
for sector_damages_csv in aggregated_summaries.glob("joined_*_aggregated.csv"):
    # Extract sector name from filename
    sector = sector_damages_csv.name.replace("joined_", "").replace("_catchments_intersection_aggregated.csv", "")
    # Only process if sector is in the water list
    if sector in water_sectors:
        print(f"Processing water sector: {sector}")
        sector_damages = pandas.read_csv(sector_damages_csv)
        future_damages = calculate_future_sector_damages(sector_damages)
        both = sector_damages.set_index('HYBAS_ID').join(future_damages)
        both.to_csv(future_damages_dir / f"future_damages_{sector}.csv")

# Now, read back the processed CSV files for the water sectors
water_damage_dfs = []
for filename in future_damages_dir.glob("future_damages*.csv"):
    # Extract sector name from filename
    sector = filename.name.replace("future_damages_", "").replace(".csv", "")
    # Only load the file if it belongs to a transport sector
    if sector in water_sectors:
        print(f"Loading water sector: {sector}")
        damages = pandas.read_csv(filename)
        damages['sector'] = sector
        water_damage_dfs.append(damages)

# Concatenate all transport sector damage dataframes into one
all_water_damage = pandas.concat(water_damage_dfs)

# Aggregate the data by 'HYBAS_ID' and calculate the sum of damages
water_rp100 = all_water_damage[
    ['HYBAS_ID', 
     'fluvial__rp_100__rcp_baseline__epoch_2010__conf_None_mean',
     'future__fluvial__rp_100__rcp_baseline__epoch_2010__conf_None_mean']
].groupby('HYBAS_ID').sum()

# Join with geometry information from hydrobasins dataframe
water_rp100 = hydrobasins[['HYBAS_ID', 'geometry']].set_index('HYBAS_ID').join(water_rp100)

# Calculate the avoided damages for the energy sector
water_rp100['avoided__fluvial__rp_100'] = (
    water_rp100['fluvial__rp_100__rcp_baseline__epoch_2010__conf_None_mean'] - 
    water_rp100['future__fluvial__rp_100__rcp_baseline__epoch_2010__conf_None_mean']
)

water_rp100.plot(column='avoided__fluvial__rp_100', legend=True, cmap='magma_r')










In [ ]:
# Define the water sectors
water_sectors = [
    "irrigation_edges",
    "pipelines_edges",
    "potable_facilities_nodes",
    "irrigation_nodes",
    "wastewater_nodes"
]

# Process only water sector CSV files
for sector_damages_csv in aggregated_summaries.glob("joined_*_aggregated.csv"):
    # Extract sector name from filename
    sector = sector_damages_csv.name.replace("joined_", "").replace("_catchments_intersection_aggregated.csv", "")
    # Only process if sector is in the water list
    if sector in water_sectors:
        print(f"Processing water sector: {sector}")
        sector_damages = pandas.read_csv(sector_damages_csv)
        future_damages = calculate_future_sector_damages(sector_damages)
        both = sector_damages.set_index('HYBAS_ID').join(future_damages)
        both.to_csv(future_damages_dir / f"future_damages_{sector}.csv")

# Now, read back the processed CSV files for the water sectors
water_damage_dfs = []
for filename in future_damages_dir.glob("future_damages*.csv"):
    # Extract sector name from filename
    sector = filename.name.replace("future_damages_", "").replace(".csv", "")
    # Only load the file if it belongs to a transport sector
    if sector in water_sectors:
        print(f"Loading water sector: {sector}")
        damages = pandas.read_csv(filename)
        damages['sector'] = sector
        water_damage_dfs.append(damages)

# Concatenate all transport sector damage dataframes into one
all_water_damage = pandas.concat(water_damage_dfs)

# Aggregate the data by 'HYBAS_ID' and calculate the sum of damages
water_rp50 = all_water_damage[
    ['HYBAS_ID', 
     'fluvial__rp_50__rcp_baseline__epoch_2010__conf_None_mean',
     'future__fluvial__rp_50__rcp_baseline__epoch_2010__conf_None_mean']
].groupby('HYBAS_ID').sum()

# Join with geometry information from hydrobasins dataframe
water_rp50 = hydrobasins[['HYBAS_ID', 'geometry']].set_index('HYBAS_ID').join(water_rp50)

# Calculate the avoided damages for the energy sector
water_rp50['avoided__fluvial__rp_50'] = (
    water_rp50['fluvial__rp_50__rcp_baseline__epoch_2010__conf_None_mean'] - 
    water_rp50['future__fluvial__rp_50__rcp_baseline__epoch_2010__conf_None_mean']
)

water_rp50.plot(column='avoided__fluvial__rp_50', legend=True, cmap='magma_r')




In [ ]:
# Define the water sectors
water_sectors = [
    "irrigation_edges",
    "pipelines_edges",
    "potable_facilities_nodes",
    "irrigation_nodes",
    "wastewater_nodes"
]

# Process only water sector CSV files
for sector_damages_csv in aggregated_summaries.glob("joined_*_aggregated.csv"):
    # Extract sector name from filename
    sector = sector_damages_csv.name.replace("joined_", "").replace("_catchments_intersection_aggregated.csv", "")
    # Only process if sector is in the water list
    if sector in water_sectors:
        print(f"Processing water sector: {sector}")
        sector_damages = pandas.read_csv(sector_damages_csv)
        future_damages = calculate_future_sector_damages(sector_damages)
        both = sector_damages.set_index('HYBAS_ID').join(future_damages)
        both.to_csv(future_damages_dir / f"future_damages_{sector}.csv")

# Now, read back the processed CSV files for the water sectors
water_damage_dfs = []
for filename in future_damages_dir.glob("future_damages*.csv"):
    # Extract sector name from filename
    sector = filename.name.replace("future_damages_", "").replace(".csv", "")
    # Only load the file if it belongs to a transport sector
    if sector in water_sectors:
        print(f"Loading water sector: {sector}")
        damages = pandas.read_csv(filename)
        damages['sector'] = sector
        water_damage_dfs.append(damages)

# Concatenate all transport sector damage dataframes into one
all_water_damage = pandas.concat(water_damage_dfs)

# Aggregate the data by 'HYBAS_ID' and calculate the sum of damages
water_rp20 = all_water_damage[
    ['HYBAS_ID', 
     'fluvial__rp_20__rcp_baseline__epoch_2010__conf_None_mean',
     'future__fluvial__rp_20__rcp_baseline__epoch_2010__conf_None_mean']
].groupby('HYBAS_ID').sum()

# Join with geometry information from hydrobasins dataframe
water_rp20 = hydrobasins[['HYBAS_ID', 'geometry']].set_index('HYBAS_ID').join(water_rp20)

# Calculate the avoided damages for the energy sector
water_rp20['avoided__fluvial__rp_20'] = (
    water_rp20['fluvial__rp_20__rcp_baseline__epoch_2010__conf_None_mean'] - 
    water_rp20['future__fluvial__rp_20__rcp_baseline__epoch_2010__conf_None_mean']
)

water_rp20.plot(column='avoided__fluvial__rp_20', legend=True, cmap='magma_r')




## 